# RAG query pipeline — weighted stratified retrieval

**Problem (corpus skew).** The unified index is dominated by English↔Japanese parallel chunks (`eng_jap_*`, tens of thousands of vectors). A single global `query_vectors` top‑k almost always returns only those rows, so **grammar** and **style** rulebooks (dozens to hundreds of chunks) rarely surface.

**Approach.** Encode the query **once**, then run **separate** `query_vectors` calls with a [metadata `filter`](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors-metadata-filtering.html) on `source_file`, so each stratum gets its own quota:

| Stratum | Top‑k | Role |
|---|---|---|
| `eng_jap` | 3 | Parallel translation examples |
| `gemini_annotated` | 1–2 (`GEMINI_TOP_K`) | Curated annotations |
| `grammar` | 1 | Structural rule |
| `style_guide` | 1 | Tone / formatting rule |

Results are merged in that order, **deduped by vector key**, then **chunk text is loaded only from the vector index**: first from metadata returned with `query_vectors`, then any gaps are filled with batched `get_vectors()` on the hit keys. **No local `kb/` JSONL reads** are used for retrieval context (an optional appendix cell below can still inspect local files for debugging).

**Legacy global hack.** A naive workaround was: wide top‑k then keep only `source_line ≤ 100`. That does **not** guarantee grammar/style coverage and is **not** used here.

**Source metadata.** `source_file` and `source_line` in vector metadata are the canonical references into the ingested JSONL (see `rag/aws_vectorDB.py`). If a stratum returns zero hits, confirm the exact `source_file` string with the optional **section 4b** probe cell below.

**Credentials:** `VECTORS_AWS_ACCESS_KEY_ID`, `VECTORS_AWS_SECRET_ACCESS_KEY`, and region (see `src/utils/aws_profiles.py`).


## 1. Dependencies (run once per environment)

In [9]:
%pip install sentence-transformers boto3 pandas --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Project path and imports

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd


def project_root() -> Path:
    """Directory that contains `src/` (works if cwd is repo root or `rag/`)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "retrieval").is_dir():
            return p
    raise RuntimeError("Could not find project root (folder with src/retrieval). cd to the repo root or rag/.")


_ROOT = project_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

import src.retrieval.s3_vectors_rag as _s3_rag
from src.retrieval.s3_vectors_rag import RetrievedChunk, format_context

print(f"Project root: {_ROOT}")

Project root: c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469


## 3. Configuration

Override with environment variables: `RAG_VECTOR_BUCKET`, `RAG_VECTOR_INDEX`, `RAG_KB_DIR`, `VECTORS_AWS_DEFAULT_REGION`, `EMBED_MODEL`, `GEMINI_TOP_K`, `RAG_MAX_CONTEXT_CHARS`, and per-corpus `RAG_SOURCE_FILE_ENG_JAP`, `RAG_SOURCE_FILE_GEMINI`, `RAG_SOURCE_FILE_GRAMMAR`, `RAG_SOURCE_FILE_STYLE` if metadata strings differ from the defaults below.

In [ ]:
# Metadata `source_file` values must match the index exactly (see §4b probe if a stratum returns 0 hits).
# Ingest uses stems like `*_chunks_embedded_full` from `rag/aws_vectorDB.py`; many indexes store `.jsonl` in metadata.
SOURCE_FILE_ENG_JAP = os.environ.get(
    "RAG_SOURCE_FILE_ENG_JAP", "eng_jap_chunks_embedded_full.jsonl"
)
SOURCE_FILE_GEMINI = os.environ.get(
    "RAG_SOURCE_FILE_GEMINI", "gemini_annotated_chunks_embedded_full.jsonl"
)
SOURCE_FILE_GRAMMAR = os.environ.get(
    "RAG_SOURCE_FILE_GRAMMAR", "grammar_chunks_embedded_full.jsonl"
)
SOURCE_FILE_STYLE = os.environ.get(
    "RAG_SOURCE_FILE_STYLE", "style_guide_chunks_embedded_full.jsonl"
)

GEMINI_TOP_K = int(os.environ.get("GEMINI_TOP_K", "2"))  # set to 1 for a single curated hit
MAX_CONTEXT_CHARS = int(os.environ.get("RAG_MAX_CONTEXT_CHARS", "12000"))
EMBED_MODEL = os.environ.get("EMBED_MODEL", "intfloat/multilingual-e5-small")

VECTOR_BUCKET = os.environ.get("RAG_VECTOR_BUCKET", "is469-genai-grp-project")
INDEX_NAME = os.environ.get("RAG_VECTOR_INDEX", "rag-vector-2")
REGION = os.environ.get("VECTORS_AWS_DEFAULT_REGION", "ap-southeast-1")

_kb_env = os.environ.get("RAG_KB_DIR", "").strip()
KB_DIR = Path(_kb_env) if _kb_env else (_ROOT / "kb")

print(f"Bucket: {VECTOR_BUCKET}")
print(f"Index:  {INDEX_NAME}")
print(f"Region: {REGION}")
print(f"kb_dir: {KB_DIR} (exists={KB_DIR.is_dir()})")
print(f"Stratified quotas: eng_jap=3, gemini={GEMINI_TOP_K}, grammar=1, style=1 | embed_model={EMBED_MODEL}")

Bucket: is469-genai-grp-project
Index:  rag-vector-2
Region: ap-southeast-1
Chunk text: S3 Vectors only (query + get_vectors); batch_size=50
Stratified quotas: eng_jap=3, gemini=2, grammar=1, style=1 | embed_model=intfloat/multilingual-e5-small


## 3b. Build S3 Vectors client for chunk text resolution

Uses `s3vectors_client` (same credentials as the retriever) to call `get_vectors()` on the keys
returned by the query. Chunk text is resolved directly from the vector metadata stored in `rag-vector-2` —
no local `kb/` file or separate S3 bucket required.


In [ ]:
from src.utils.aws_profiles import s3vectors_client

# Same bucket + index as the retriever
vectors_client = s3vectors_client(region_name=REGION)

def resolve_chunks_from_s3(keys: list[str]) -> dict[str, str]:
    """
    Fetch full vector metadata for a list of keys from rag-vector-2.
    Returns a dict of {key: chunk_text}.
    """
    if not keys:
        return {}
    response = vectors_client.get_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=INDEX_NAME,
        keys=keys,
        returnData=False,       # We only need metadata, not the embedding floats
        returnMetadata=True,
    )
    result = {}
    for vec in response.get("vectors", []):
        key = vec.get("key")
        meta = vec.get("metadata") or {}
        # Try common field names the ingest script may have used
        text = (
            meta.get("text")
            or meta.get("chunk_text")
            or meta.get("content")
        )
        if key and text:
            result[key] = text
    return result

print("✓ s3vectors_client ready — chunk text will be resolved from rag-vector-2 metadata")


✓ s3vectors_client ready — chunk text from S3 Vectors metadata only (batched get_vectors)


## 4. KB path mapping and query embedder

The next cell patches `s3_vectors_rag._guess_kb_paths` so metadata names match **this repo’s** `kb/` files, and defines a lazy `SentenceTransformer` encoder (same `query: ` prefix and model as `S3VectorsRAGRetriever` in `s3_vectors_rag.py`).

In [ ]:
_embedder = None


def get_embedder():
    """Lazy-load the same E5 query encoder convention as `S3VectorsRAGRetriever`."""
    global _embedder
    if _embedder is None:
        import torch
        from sentence_transformers import SentenceTransformer

        device = os.environ.get("RAG_EMBED_DEVICE", "cpu")
        if device == "cuda" and not torch.cuda.is_available():
            device = "cpu"
        _embedder = SentenceTransformer(EMBED_MODEL, trust_remote_code=True, device=device)
        _embedder.max_seq_length = int(
            os.environ.get("MAX_SEQ_LENGTH", "512" if device == "cpu" else "1024")
        )
    return _embedder


def encode_query_vector(query_en: str) -> list[float]:
    import numpy as np

    model = get_embedder()
    text = query_en if query_en.lower().startswith("query:") else f"query: {query_en}"
    vec = model.encode(text, convert_to_numpy=True, show_progress_bar=False)
    return vec.astype("float32").tolist()


print("Query embedder ready (model loads on first `encode_query_vector`).")

Query embedder ready (model loads on first `encode_query_vector`).


## 5. Weighted stratified retrieval

Set `USER_QUERY` to the English sentence (or student question), then run the next cell. It runs **four** filtered `query_vectors` calls (optionally in parallel), merges by quota order, resolves text, and builds `hits_df`.

In [26]:
from concurrent.futures import ThreadPoolExecutor

USER_QUERY = "Example: How do I express polite requests in Japanese?"


S3_TEXT_MISSING_PREFIX = "(no chunk text in S3 vector metadata for key="


def _vectors_response_to_chunks(items: list) -> list[RetrievedChunk]:
    """Build chunks from query hits; text comes from query metadata only (get_vectors fills gaps later)."""
    out: list[RetrievedChunk] = []
    for item in items:
        key = str(item.get("key", ""))
        distance = item.get("distance")
        if distance is not None:
            try:
                distance = float(distance)
            except (TypeError, ValueError):
                distance = None
        meta = item.get("metadata") or {}
        source_file = str(meta.get("source_file", ""))
        try:
            source_line = int(meta.get("source_line", -1))
        except (TypeError, ValueError):
            source_line = -1
        text = _text_from_vector_metadata(meta)
        out.append(
            RetrievedChunk(
                key=key,
                distance=distance,
                source_file=source_file,
                source_line=source_line,
                text=text,
            )
        )
    return out


STRATA_SPECS = [
    {"name": "eng_jap", "source_file": SOURCE_FILE_ENG_JAP, "top_k": 3},
    {"name": "gemini_annotated", "source_file": SOURCE_FILE_GEMINI, "top_k": GEMINI_TOP_K},
    {"name": "grammar", "source_file": SOURCE_FILE_GRAMMAR, "top_k": 1},
    {"name": "style_guide", "source_file": SOURCE_FILE_STYLE, "top_k": 1},
]

STRATA_MERGE_ORDER = ["eng_jap", "gemini_annotated", "grammar", "style_guide"]


def _query_one_stratum(spec: dict) -> tuple[str, list[RetrievedChunk]]:
    name = spec["name"]
    sf = spec["source_file"]
    k = int(spec["top_k"])
    flt = {"source_file": {"$eq": sf}}
    try:
        resp = vectors_client.query_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=INDEX_NAME,
            topK=k,
            queryVector={"float32": qvec},
            returnMetadata=True,
            returnDistance=True,
            filter=flt,
        )
    except Exception as e:
        print(f"[{name}] query_vectors failed ({sf!r}): {e}")
        return name, []
    vecs = resp.get("vectors") or []
    if not vecs:
        print(f"[{name}] warning: 0 hits for source_file={sf!r} — check SOURCE_FILE_* / unfiltered query sample.")
    chs = _vectors_response_to_chunks(vecs)
    print(f"[{name}] retrieved {len(chs)} chunk(s) (requested top_k={k}).")
    return name, chs


qvec = encode_query_vector(USER_QUERY)

with ThreadPoolExecutor(max_workers=4) as ex:
    stratum_results = list(ex.map(_query_one_stratum, STRATA_SPECS))

by_name = dict(stratum_results)
chunks: list[RetrievedChunk] = []
chunk_stratum: dict[str, str] = {}
seen_keys: set[str] = set()
for nm in STRATA_MERGE_ORDER:
    for c in by_name.get(nm, []):
        if c.key and c.key not in seen_keys:
            seen_keys.add(c.key)
            chunks.append(c)
            chunk_stratum[c.key] = nm

# ── Fill missing text via get_vectors (S3 only; no local JSONL) ───────────────
keys_missing_text = [c.key for c in chunks if c.key and not (c.text or "").strip()]
if keys_missing_text:
    print(f"\nFetching text for {len(keys_missing_text)} chunk(s) via get_vectors...")
    s3_text_map = resolve_chunks_from_s3(keys_missing_text)
    for chunk in chunks:
        if chunk.key and not (chunk.text or "").strip():
            chunk.text = s3_text_map.get(chunk.key, "")

for chunk in chunks:
    if chunk.key and not (chunk.text or "").strip():
        chunk.text = (
            f"{S3_TEXT_MISSING_PREFIX}{chunk.key}; "
            f"source_file={chunk.source_file} line={chunk.source_line})"
        )

context = format_context(chunks, max_chars=MAX_CONTEXT_CHARS)

print("\n=== Formatted context (for prompting) ===\n")
print(context)

# ── DataFrame summary ────────────────────────────────────────────────────────
_preview_chars = 320
_rows: list[dict] = []
for i, c in enumerate(chunks, start=1):
    unresolved = (c.text or "").startswith("(no local text resolved")
    t = c.text or ""
    preview = t if len(t) <= _preview_chars else f"{t[: _preview_chars - 1]}…"
    _rows.append(
        {
            "rank": i,
            "stratum": chunk_stratum.get(c.key, ""),
            "distance": c.distance,
            "key": c.key,
            "source_file": c.source_file,
            "source_line": c.source_line,
            "resolved": not unresolved,
            "text_preview": preview,
            "chunk_text": t,
        }
    )

hits_df = pd.DataFrame(_rows)
if hits_df["distance"].notna().any():
    hits_df["distance"] = pd.to_numeric(hits_df["distance"], errors="coerce").round(6)

hits_summary = hits_df[
    ["rank", "stratum", "distance", "key", "source_file", "source_line", "resolved", "text_preview"]
]
print("\n=== Retrieval hits (summary DataFrame: `hits_df`, display below) ===\n")
hits_summary


[gemini_annotated] retrieved 2 chunk(s) (requested top_k=2).
[eng_jap] retrieved 3 chunk(s) (requested top_k=3).
[style_guide] warning: 0 hits for source_file='style_guide_chunks_embedded_full.jsonl' — check SOURCE_FILE_* / unfiltered query sample.
[style_guide] retrieved 0 chunk(s) (requested top_k=1).
[grammar] retrieved 1 chunk(s) (requested top_k=1).

Fetching text for 6 chunk(s) via get_vectors...

=== Formatted context (for prompting) ===

[eng_jap_chunks_embedded_full.jsonl L8432]
(no chunk text in S3 vector metadata for key=eng_jap_chunks_embedded_full:8431; source_file=eng_jap_chunks_embedded_full.jsonl line=8432)

---

[eng_jap_chunks_embedded_full.jsonl L79]
(no chunk text in S3 vector metadata for key=eng_jap_chunks_embedded_full:78; source_file=eng_jap_chunks_embedded_full.jsonl line=79)

---

[eng_jap_chunks_embedded_full.jsonl L30252]
(no chunk text in S3 vector metadata for key=eng_jap_chunks_embedded_full:30251; source_file=eng_jap_chunks_embedded_full.jsonl line=30252

,rank,stratum,distance,key,source_file,source_line,resolved,text_preview
0,1,eng_jap,0.103171,eng_jap_chunks_embedded_full:8431,eng_jap_chunks_embedded_full.jsonl,8432,True,(no chunk text in S3 vector metadata for key=e...
1,2,eng_jap,0.104292,eng_jap_chunks_embedded_full:78,eng_jap_chunks_embedded_full.jsonl,79,True,(no chunk text in S3 vector metadata for key=e...
2,3,eng_jap,0.105754,eng_jap_chunks_embedded_full:30251,eng_jap_chunks_embedded_full.jsonl,30252,True,(no chunk text in S3 vector metadata for key=e...
3,4,gemini_annotated,0.120441,gemini_annotated_chunks_embedded_full:410,gemini_annotated_chunks_embedded_full.jsonl,411,True,(no chunk text in S3 vector metadata for key=g...
4,5,gemini_annotated,0.124012,gemini_annotated_chunks_embedded_full:101,gemini_annotated_chunks_embedded_full.jsonl,102,True,(no chunk text in S3 vector metadata for key=g...
5,6,grammar,0.163426,grammar_chunks_embedded_full:29,grammar_chunks_embedded_full.jsonl,30,True,(no chunk text in S3 vector metadata for key=g...


In [27]:
# Source-line sanity check (Eng↔Jap skew + why local resolution disagrees with labels)
import json

chunks_file = _ROOT / "kb" / "eng_jap_chunks.jsonl"
print(f"{chunks_file.name} exists: {chunks_file.exists()}")

with open(chunks_file, encoding="utf-8") as f:
    lines = f.readlines()

print(
    f"Total JSONL lines: {len(lines):,} — global top-k is dominated by this file; "
    "this notebook uses per-corpus filtered retrieval instead of a line band.\n"
)

print("First rows in the early band (typical context after filtering):")
for lineno in range(1, min(4, len(lines) + 1)):
    record = json.loads(lines[lineno - 1])
    snippet = json.dumps(record, ensure_ascii=False)[:180]
    print(f"  L{lineno}: {snippet}…")

# Deep rows exist on disk; "unresolved" in the retriever is usually path/name mismatch
# (metadata says eng_jap_chunks_embedded_full.jsonl but repo line lives in eng_jap_chunks.jsonl).
for lineno in [8432, 30252]:
    if lineno <= len(lines):
        record = json.loads(lines[lineno - 1])
        snippet = json.dumps(record, ensure_ascii=False)[:160]
        print(f"\nDeep row still valid at L{lineno} in eng_jap_chunks.jsonl: {snippet}…")

eng_jap_chunks.jsonl exists: True
Total JSONL lines: 46,616 — global top-k is dominated by this file; this notebook uses per-corpus filtered retrieval instead of a line band.

First rows in the early band (typical context after filtering):
  L1: {"chunk_id": "engjap_chunk_000000_000007", "start_row": 0, "end_row": 7, "n_pairs": 8, "chunk_text": "[1] EN: Let's try something.\n[1] JA: 何かしてみましょう。\n\n[2] EN: I have to go to sl…
  L2: {"chunk_id": "engjap_chunk_000006_000013", "start_row": 6, "end_row": 13, "n_pairs": 8, "chunk_text": "[1] EN: The password is \"Muiriel\".\n[1] JA: パスワードは「Muiriel」です。\n\n[2] EN: I…
  L3: {"chunk_id": "engjap_chunk_000012_000019", "start_row": 12, "end_row": 19, "n_pairs": 8, "chunk_text": "[1] EN: I will be back soon.\n[1] JA: すぐ戻る。\n\n[2] EN: I'm at a loss for wor…

Deep row still valid at L8432 in eng_jap_chunks.jsonl: {"chunk_id": "engjap_chunk_050586_050593", "start_row": 50586, "end_row": 50593, "n_pairs": 8, "chunk_text": "[1] EN: I'm very pleased to me